# Data Setup

## Import packages and CSV files

In [40]:
from pathlib import Path
import json
import time
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd
import numpy as np

SEASON_START = 2014
SEASON_END = 2025

data_directory = Path("../data")
OPENMETEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_CACHE = data_directory / "openmeteo_race_weather.csv"

circuits = pd.read_csv(data_directory / "circuits.csv", na_values=[r"\N"])
constructor_standings = pd.read_csv(data_directory / "constructor_standings.csv", na_values=[r"\N"])
constructors = pd.read_csv(data_directory / "constructors.csv", na_values=[r"\N"])
driver_standings = pd.read_csv(data_directory / "driver_standings.csv", na_values=[r"\N"])
drivers = pd.read_csv(data_directory / "drivers.csv", na_values=[r"\N"])
qualifying = pd.read_csv(data_directory / "qualifying.csv", na_values=[r"\N"])
races = pd.read_csv(data_directory / "races.csv", na_values=[r"\N"])
results = pd.read_csv(data_directory / "results.csv", na_values=[r"\N"])
safety_cars = pd.read_csv(data_directory / "safety_cars.csv", na_values=[r"\N"])
red_flags = pd.read_csv(data_directory / "red_flags.csv", na_values=[r"\N"])

## Helper methods

In [41]:
def parse_lap_time(t):
    if pd.isna(t) or t in (0, "0", "\\N", ""):
        return pd.NaT
    t_str = str(t).strip()
    if t_str.count(":") == 1:
        t_str = "00:" + t_str
    return pd.to_timedelta(t_str, errors="coerce")


def is_dnf(position_text):
    if pd.isna(position_text):
        return 0
    return int(str(position_text).isalpha())


def local_start_hour(utc_time, lng):
    if pd.isna(utc_time):
        return np.nan
    if isinstance(utc_time, str):
        utc_time = pd.to_timedelta(utc_time)
    utc_hours = utc_time.total_seconds() / 3600
    return (utc_hours + lng / 15.0) % 24


def time_of_day(hour):
    if pd.isna(hour):
        return pd.NA
    if 8 <= hour < 17:
        return "day"
    return "night"


def prev_mean(series, window=None):
    shifted = series.shift(1)
    if window is None:
        return shifted.expanding().mean()
    return shifted.rolling(window, min_periods=1).mean()


def event_key(year, name):
    return year.astype(str) + " " + name.astype(str)


def _utc_start_hour(utc_time):
    if pd.isna(utc_time):
        return np.nan
    if isinstance(utc_time, str):
        utc_time = pd.to_timedelta(utc_time)
    return int(round(utc_time.total_seconds() / 3600)) % 24


def _openmeteo_hour(lat, lng, date_str, hour, retries=4):
    params = urllib.parse.urlencode(
        {
            "latitude": lat,
            "longitude": lng,
            "start_date": date_str,
            "end_date": date_str,
            "hourly": "temperature_2m,wind_speed_10m,relative_humidity_2m,rain",
            "timezone": "UTC",
        }
    )
    url = f"{OPENMETEO_ARCHIVE}?{params}"
    last_err = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "f1-race-prediction/cs6375"}
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                payload = json.loads(resp.read().decode())
            hourly = payload.get("hourly") or {}
            times = hourly.get("time") or []
            target = f"{date_str}T{int(hour):02d}:00"
            if target in times:
                i = times.index(target)
            elif times:
                i = min(
                    range(len(times)),
                    key=lambda j: abs(pd.Timestamp(times[j]).hour - int(hour)),
                )
            else:
                return None
            rain_mm = hourly.get("rain", [None])[i]
            return {
                "air_temp": hourly.get("temperature_2m", [None])[i],
                "wind_speed": hourly.get("wind_speed_10m", [None])[i],
                "humidity": hourly.get("relative_humidity_2m", [None])[i],
                "rain": np.nan if rain_mm is None else int(float(rain_mm) > 0),
            }
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, json.JSONDecodeError) as err:
            last_err = err
            time.sleep(1.5 * (attempt + 1))
    print(f"Open-Meteo failed for {date_str} {hour:02d}:00 ({lat},{lng}): {last_err}")
    return None


def load_or_fetch_race_weather(race_keys):
    cache_cols = ["raceId", "air_temp", "wind_speed", "humidity", "rain"]
    cache = pd.DataFrame(columns=cache_cols)
    if WEATHER_CACHE.exists():
        cache = pd.read_csv(WEATHER_CACHE)
        cache = cache[cache_cols]

    rows = []
    fetched = 0
    for rec in race_keys.itertuples(index=False):
        hit = cache.loc[cache["raceId"] == rec.raceId]
        if len(hit) and hit[cache_cols[1:]].notna().all(axis=None):
            rows.append(hit.iloc[0].to_dict())
            continue
        hour = _utc_start_hour(rec.race_utc_time)
        date_str = pd.Timestamp(rec.date).strftime("%Y-%m-%d")
        if pd.isna(hour) or pd.isna(rec.lat) or pd.isna(rec.lng):
            rows.append({"raceId": rec.raceId, "air_temp": np.nan, "wind_speed": np.nan, "humidity": np.nan, "rain": np.nan})
            continue
        wx = _openmeteo_hour(rec.lat, rec.lng, date_str, hour)
        if wx is None:
            rows.append({"raceId": rec.raceId, "air_temp": np.nan, "wind_speed": np.nan, "humidity": np.nan, "rain": np.nan})
        else:
            rows.append({"raceId": rec.raceId, **wx})
            fetched += 1
            time.sleep(0.1)

    weather = pd.DataFrame(rows)
    weather.to_csv(WEATHER_CACHE, index=False)
    print(f"Weather: {fetched} Open-Meteo fetches, {weather['air_temp'].notna().sum()} / {len(weather)} races filled")
    return weather

## Data pipeline

### Coerce types and dates

In [42]:
# numeric coercions
results = results.copy()
results["grid"] = pd.to_numeric(results["grid"], errors="coerce")
results["positionOrder"] = pd.to_numeric(results["positionOrder"], errors="coerce")
results["points"] = pd.to_numeric(results["points"], errors="coerce")
qualifying = qualifying.copy()
qualifying["position"] = pd.to_numeric(qualifying["position"], errors="coerce")
driver_standings["points"] = pd.to_numeric(driver_standings["points"], errors="coerce")
driver_standings["position"] = pd.to_numeric(driver_standings["position"], errors="coerce")
driver_standings["wins"] = pd.to_numeric(driver_standings["wins"], errors="coerce")
constructor_standings["points"] = pd.to_numeric(constructor_standings["points"], errors="coerce")
constructor_standings["position"] = pd.to_numeric(constructor_standings["position"], errors="coerce")
constructor_standings["wins"] = pd.to_numeric(constructor_standings["wins"], errors="coerce")

races = races.copy()
races["date"] = pd.to_datetime(races["date"])
races["event_key"] = event_key(races["year"], races["name"])
drivers = drivers.copy()
drivers["dob"] = pd.to_datetime(drivers["dob"])

# previous race in the same season (standings are after each race)
race_keys = races[["raceId", "year", "round"]].rename(columns={"raceId": "prev_race_id", "round": "prev_round"})
prev_map = races[["raceId", "year", "round"]].copy()
prev_map["prev_round"] = prev_map["round"] - 1
prev_map = prev_map.merge(race_keys, on=["year", "prev_round"], how="left")[["raceId", "prev_race_id"]]

### Builds historical form data

In [43]:
# all-era result history, sorted in race order
hist = results.drop(columns=["time"], errors="ignore").merge(
    races[["raceId", "year", "round", "circuitId", "date", "name", "time"]].rename(
        columns={"time": "race_utc_time"}
    ),
    on="raceId",
    how="left",
)
hist = hist.sort_values(["date", "raceId", "driverId"]).reset_index(drop=True)
hist["dnf_flag"] = hist["positionText"].map(is_dnf)
hist["win"] = (hist["positionOrder"] == 1).astype(int)
hist["podium"] = (hist["positionOrder"] <= 3).astype(int)
hist["places_gained"] = hist["grid"] - hist["positionOrder"]

hist = hist.sort_values(["driverId", "date", "raceId"])
g_driver = hist.groupby("driverId", sort=False)
hist["career_races"] = g_driver.cumcount()
hist["career_wins"] = g_driver["win"].cumsum() - hist["win"]
hist["career_podiums"] = g_driver["podium"].cumsum() - hist["podium"]
hist["career_avg_finish"] = np.where(
    hist["career_races"] > 0,
    (g_driver["positionOrder"].cumsum() - hist["positionOrder"]) / hist["career_races"],
    np.nan,
)
hist["career_dnf_rate"] = g_driver["dnf_flag"].transform(lambda s: prev_mean(s))

g_circuit = hist.groupby(["driverId", "circuitId"], sort=False)
hist["circuit_races"] = g_circuit.cumcount()
hist["circuit_wins"] = g_circuit["win"].cumsum() - hist["win"]
hist["circuit_podiums"] = g_circuit["podium"].cumsum() - hist["podium"]
hist["circuit_avg_finish"] = np.where(
    hist["circuit_races"] > 0,
    (g_circuit["positionOrder"].cumsum() - hist["positionOrder"]) / hist["circuit_races"],
    np.nan,
)
hist["circuit_dnf_rate"] = g_circuit["dnf_flag"].transform(lambda s: prev_mean(s))

g_season = hist.groupby(["year", "driverId"], sort=False)
hist["avg_finish_l3"] = g_season["positionOrder"].transform(lambda s: prev_mean(s, window=3))
hist["season_dnf_rate"] = g_season["dnf_flag"].transform(lambda s: prev_mean(s))

hist["prev_constructor"] = g_driver["constructorId"].shift(1)
new_stint = hist["prev_constructor"].ne(hist["constructorId"]) | hist["prev_constructor"].isna()
hist["stint_id"] = new_stint.groupby(hist["driverId"]).cumsum()
hist["stint_races_with_constructor"] = hist.groupby(["driverId", "stint_id"]).cumcount()
hist["races_with_constructor"] = hist.groupby(["driverId", "constructorId"]).cumcount()

# constructor form: mean finish of both cars, last 3 races this season
ctor_race = (
    hist.groupby(["year", "constructorId", "raceId", "date"], as_index=False)
    .agg(ctor_mean_finish=("positionOrder", "mean"))
    .sort_values(["constructorId", "year", "date", "raceId"])
)
ctor_race["constructor_avg_finish_l3"] = ctor_race.groupby(["year", "constructorId"])[
    "ctor_mean_finish"
].transform(lambda s: prev_mean(s, window=3))

# circuit overtaking proxy: mean (grid - finish) at this circuit in earlier races
circuit_gain = (
    hist.groupby(["circuitId", "raceId", "date"], as_index=False)
    .agg(race_position_gain=("places_gained", "mean"))
    .sort_values(["circuitId", "date", "raceId"])
)
circuit_gain["circuit_position_gain_avg"] = circuit_gain.groupby("circuitId")[
    "race_position_gain"
].transform(lambda s: prev_mean(s))

### Computes qualifying gaps

In [44]:
# qualifying: last session time minus fastest time in that same session
quali = qualifying.copy()
for col in ["q1", "q2", "q3"]:
    quali[col] = quali[col].map(parse_lap_time)

session_fastest = quali.groupby("raceId")[["q1", "q2", "q3"]].transform("min")
quali["last_quali"] = quali["q3"].fillna(quali["q2"]).fillna(quali["q1"])
quali["last_quali_session"] = np.select(
    [quali["q3"].notna(), quali["q2"].notna(), quali["q1"].notna()],
    [3, 2, 1],
    default=np.nan,
)
quali["session_fastest"] = session_fastest["q3"].where(
    quali["q3"].notna(),
    session_fastest["q2"].where(quali["q2"].notna(), session_fastest["q1"]),
)
quali["quali_gap_to_pole_s"] = (
    (quali["last_quali"] - quali["session_fastest"]).dt.total_seconds().clip(lower=0)
)

quali["best_quali_s"] = quali["last_quali"].dt.total_seconds()
quali["n_teammate_times"] = quali.groupby(["raceId", "constructorId"])["best_quali_s"].transform("count")
quali["teammate_sum_s"] = quali.groupby(["raceId", "constructorId"])["best_quali_s"].transform("sum")
quali["teammate_quali_gap_s"] = np.where(
    (quali["n_teammate_times"] > 1) & quali["best_quali_s"].notna(),
    quali["best_quali_s"] - (quali["teammate_sum_s"] - quali["best_quali_s"]) / (quali["n_teammate_times"] - 1),
    np.nan,
)
quali_features = quali[
    ["raceId", "driverId", "position", "last_quali_session", "quali_gap_to_pole_s", "teammate_quali_gap_s"]
].rename(columns={"position": "quali_position"})

### Circuit events

In [45]:
# safety cars and red flags: historical rates at this circuit, excluding the current race
sc = safety_cars.copy()
sc["event_key"] = sc["Race"].astype(str)
sc = sc.merge(races[["event_key", "raceId"]], on="event_key", how="left")
sc_counts = sc.dropna(subset=["raceId"]).groupby("raceId").size().rename("sc_count")

rf = red_flags.copy()
rf["event_key"] = rf["Race"].astype(str)
rf = rf.merge(races[["event_key", "raceId"]], on="event_key", how="left")
rf_counts = rf.dropna(subset=["raceId"]).groupby("raceId").size().rename("rf_count")

race_events = races[["raceId", "circuitId", "date"]].merge(sc_counts, on="raceId", how="left")
race_events = race_events.merge(rf_counts, on="raceId", how="left")
race_events["sc_count"] = race_events["sc_count"].fillna(0)
race_events["rf_count"] = race_events["rf_count"].fillna(0)
race_events["has_sc"] = (race_events["sc_count"] > 0).astype(int)
race_events["has_rf"] = (race_events["rf_count"] > 0).astype(int)
race_events = race_events.sort_values(["circuitId", "date", "raceId"])
g_ce = race_events.groupby("circuitId")
race_events["circuit_safety_car_rate"] = g_ce["has_sc"].transform(lambda s: prev_mean(s)).fillna(0)
race_events["circuit_safety_car_avg"] = g_ce["sc_count"].transform(lambda s: prev_mean(s)).fillna(0)
race_events["circuit_red_flag_rate"] = g_ce["has_rf"].transform(lambda s: prev_mean(s)).fillna(0)

### Builds the final feature table

In [46]:
features = hist.loc[
    (hist["year"] >= SEASON_START) & (hist["year"] <= SEASON_END)
].copy()

features = features.merge(prev_map, on="raceId", how="left")
features = features.merge(
    driver_standings[["raceId", "driverId", "points", "position", "wins"]].rename(
        columns={
            "raceId": "prev_race_id",
            "points": "driver_champ_points",
            "position": "driver_champ_pos",
            "wins": "driver_champ_wins",
        }
    ),
    on=["prev_race_id", "driverId"],
    how="left",
)
features = features.merge(
    constructor_standings[["raceId", "constructorId", "points", "position", "wins"]].rename(
        columns={
            "raceId": "prev_race_id",
            "points": "constructor_champ_points",
            "position": "constructor_champ_pos",
            "wins": "constructor_champ_wins",
        }
    ),
    on=["prev_race_id", "constructorId"],
    how="left",
)
standings_cols = [
    "driver_champ_points",
    "driver_champ_pos",
    "driver_champ_wins",
    "constructor_champ_points",
    "constructor_champ_pos",
    "constructor_champ_wins",
]
features[standings_cols] = features[standings_cols].fillna(0)

features = features.merge(quali_features, on=["raceId", "driverId"], how="left")
features = features.merge(
    ctor_race[["raceId", "constructorId", "constructor_avg_finish_l3"]],
    on=["raceId", "constructorId"],
    how="left",
)
features = features.merge(
    circuit_gain[["raceId", "circuit_position_gain_avg"]],
    on="raceId",
    how="left",
)
features = features.merge(
    race_events[
        ["raceId", "circuit_safety_car_rate", "circuit_safety_car_avg", "circuit_red_flag_rate"]
    ],
    on="raceId",
    how="left",
)
features = features.merge(
    circuits[["circuitId", "circuitRef", "circuitType", "alt", "lat", "lng"]].rename(
        columns={"circuitRef": "circuit_ref", "circuitType": "circuit_type", "alt": "circuit_alt"}
    ),
    on="circuitId",
    how="left",
)
features = features.merge(
    drivers[["driverId", "driverRef", "dob"]].rename(columns={"driverRef": "driver_ref"}),
    on="driverId",
    how="left",
)
features = features.merge(
    constructors[["constructorId", "constructorRef"]].rename(
        columns={"constructorRef": "constructor_ref"}
    ),
    on="constructorId",
    how="left",
)

features["driver_age_years"] = (features["date"] - features["dob"]).dt.days / 365.25
features["grid"] = features["grid"].fillna(features["quali_position"])
features["grid_minus_quali"] = features["grid"] - features["quali_position"]
features["race_time_of_day"] = [
    time_of_day(local_start_hour(t, lng))
    for t, lng in zip(features["race_utc_time"], features["lng"])
]
race_weather_keys = features[["raceId", "date", "race_utc_time", "lat", "lng"]].drop_duplicates("raceId")
weather = load_or_fetch_race_weather(race_weather_keys)
features = features.merge(weather, on="raceId", how="left")
features["row_id"] = features["raceId"].astype(str) + "_" + features["driverId"].astype(str)
features["finish_position"] = features["positionOrder"]
features["dnf"] = features["dnf_flag"]
features["race_name"] = features["name"]

Weather: 0 Open-Meteo fetches, 252 / 252 races filled


## Export CSV

In [47]:
ID_COLS = [
    "row_id",
    "raceId",
    "year",
    "round",
    "date",
    "race_name",
    "driverId",
    "driver_ref",
    "constructorId",
    "constructor_ref",
    "circuitId",
    "circuit_ref",
]

FEATURE_COLS = [
    "grid",
    "quali_position",
    "last_quali_session",
    "quali_gap_to_pole_s",
    "teammate_quali_gap_s",
    "grid_minus_quali",
    "driver_champ_pos",
    "driver_champ_points",
    "driver_champ_wins",
    "constructor_champ_pos",
    "constructor_champ_points",
    "constructor_champ_wins",
    "constructor_avg_finish_l3",
    "career_races",
    "career_wins",
    "career_podiums",
    "career_avg_finish",
    "career_dnf_rate",
    "circuit_races",
    "circuit_avg_finish",
    "circuit_wins",
    "circuit_podiums",
    "races_with_constructor",
    "stint_races_with_constructor",
    "avg_finish_l3",
    "season_dnf_rate",
    "circuit_dnf_rate",
    "driver_age_years",
    "circuit_type",
    "circuit_alt",
    "circuit_position_gain_avg",
    "circuit_safety_car_rate",
    "circuit_safety_car_avg",
    "circuit_red_flag_rate",
    "race_time_of_day",
    "air_temp",
    "wind_speed",
    "humidity",
    "rain",
]

LABEL_COLS = ["finish_position", "dnf"]

output_cols = ID_COLS + FEATURE_COLS + LABEL_COLS
driver_race = features[output_cols].sort_values(["date", "raceId", "grid", "driverId"]).reset_index(drop=True)

out_path = data_directory / "driver_race_features.csv"
driver_race.to_csv(out_path, index=False)
print(f"Wrote {len(driver_race)} rows to {out_path}")

Wrote 5105 rows to ..\data\driver_race_features.csv


## Check

In [48]:
assert driver_race["row_id"].is_unique
assert set(LABEL_COLS).isdisjoint(FEATURE_COLS)
print("rows", len(driver_race), "races", driver_race["raceId"].nunique())
print("years", driver_race.groupby("year")["raceId"].nunique().to_dict())
print("null rates")
print(driver_race.isna().mean().sort_values(ascending=False).head(15).round(3))

r1 = driver_race[(driver_race["year"] == 2014) & (driver_race["round"] == 1)]
print("2014 R1 standings max (expect 0)", r1[standings_cols].max().to_dict())
print("2014 R1 avg_finish_l3 all NA", r1["avg_finish_l3"].isna().all())

ham = driver_race[(driver_race["driver_ref"] == "hamilton") & (driver_race["year"] == 2014) & (driver_race["round"] == 1)]
print("quali gap min (expect >= 0)", driver_race["quali_gap_to_pole_s"].min())
print(ham[["row_id", "race_name", "grid", "quali_position", "quali_gap_to_pole_s", "career_races", "career_wins", "finish_position", "dnf", "air_temp", "wind_speed", "humidity", "rain"]])

rain_ok = driver_race["rain"].dropna()
assert rain_ok.isin([0, 1]).all()
print("weather filled races", driver_race.groupby("raceId")["air_temp"].first().notna().mean())
print("rain share", float(rain_ok.mean()))

rows 5105 races 252
years {2014: 19, 2015: 19, 2016: 21, 2017: 20, 2018: 21, 2019: 21, 2020: 17, 2021: 22, 2022: 22, 2023: 22, 2024: 24, 2025: 24}
null rates
circuit_avg_finish           0.205
circuit_dnf_rate             0.205
season_dnf_rate              0.052
avg_finish_l3                0.052
constructor_avg_finish_l3    0.048
circuit_position_gain_avg    0.032
teammate_quali_gap_s         0.028
last_quali_session           0.014
quali_gap_to_pole_s          0.014
career_avg_finish            0.008
career_dnf_rate              0.008
quali_position               0.003
grid_minus_quali             0.003
date                         0.000
round                        0.000
dtype: float64
2014 R1 standings max (expect 0) {'driver_champ_points': 0.0, 'driver_champ_pos': 0.0, 'driver_champ_wins': 0.0, 'constructor_champ_points': 0.0, 'constructor_champ_pos': 0.0, 'constructor_champ_wins': 0.0}
2014 R1 avg_finish_l3 all NA True
quali gap min (expect >= 0) 0.0
  row_id              race_na